# Word Rate Plotter
Plots the relative frequency (rate) of words over time from CCAnalysis JSON output files.

**JSON format expected:**
```json
{"total_word_occurrences": 1049088412, "words": [{"word": "fire", "count": 252855133}, ...]}
```

In [ ]:
# Install dependencies if needed (Colab usually has these)
# !pip install matplotlib

In [ ]:
import json
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────────────────

# Map each JSON file to the date it represents.
# Date strings can be any unambiguous format: "2013-01", "2020-06-15", etc.
FILES = {
    "counts_2013_20.json": "2013-01-01",
    "counts_2020_10.json": "2020-01-01",
    # "counts_2020_20.json": "2020-01-01",
}

# Words to plot (must appear in the JSON files)
WORDS_FILE = "target_words.txt"

# Set to True to upload files interactively in Colab instead of using paths above
USE_COLAB_UPLOAD = False
# ───────────────────────────────────────────────────────────────────────────────

In [ ]:
def load_counts(path):
    """Load a CCAnalysis JSON file and return {word: rate} using total_word_occurrences."""
    with open(path) as f:
        data = json.load(f)
    total = data["total_word_occurrences"]
    return {entry["word"]: entry["count"] / total for entry in data["words"]}


def parse_date(s):
    for fmt in ("%Y-%m-%d", "%Y-%m", "%Y"):
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            continue
    raise ValueError(f"Cannot parse date: {s!r}")

In [ ]:
if USE_COLAB_UPLOAD:
    from google.colab import files
    print("Upload your JSON files, then enter a date for each when prompted.")
    uploaded = files.upload()  # dict of {filename: bytes}
    import io
    file_map = {}
    for name in uploaded:
        date_str = input(f"Date for '{name}' (e.g. 2020-06): ")
        file_map[name] = (io.BytesIO(uploaded[name]), date_str)
else:
    file_map = {path: (path, date_str) for path, date_str in FILES.items()}

# Build timeline: sorted list of (datetime, {word: rate})
timeline = []
for path, (source, date_str) in file_map.items():
    if USE_COLAB_UPLOAD:
        import json as _json
        data = _json.load(source)
        total = data["total_word_occurrences"]
        rates = {e["word"]: e["count"] / total for e in data["words"]}
    else:
        rates = load_counts(source)
    timeline.append((parse_date(date_str), rates))

timeline.sort(key=lambda x: x[0])
dates = [t for t, _ in timeline]
print(f"Loaded {len(timeline)} file(s): {[d.strftime('%Y-%m-%d') for d in dates]}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

words = []
with open(WORDS_FILE) as word_file:
    for line in word_file:
        if "#" not in line:
            words.append(line)

for word in words:
    rates = [r.get(word, 0) for _, r in timeline]
    if any(r > 0 for r in rates):
        ax.plot(dates, rates, marker="o", label=word)
    else:
        print(f"Warning: '{word}' not found in any file — skipping.")

ax.set_title("Word Rate Over Time")
ax.set_xlabel("Date")
ax.set_ylabel("Relative frequency (count / total words)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
fig.autofmt_xdate()
ax.legend()
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()